#### 开始

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import utils_z
import cityjson_parser as cjpar

In [3]:
conn = utils_z.get_conn("Test20260413", "postgres", "we6666", "localhost", "5432")

In [30]:
block_table_name = "amsterdam" + "_blocks"
lod1_table_name = "amsterdam" + "_buildings_lod1"
lod2_table_name = "amsterdam" + "_buildings_lod2"
lod2_surface_table_name  = "amsterdam" + "_building_surfaces_lod2"
city_prefix  = "NL_AM"
target_srid  = 4326
source_srid  = 28992

#### 大阪数据检查

In [10]:
citygml_tools_path = r"E:\0_mylib\citygml-tools-2.4.1\citygml-tools.bat"

utils_z.run_cmd(f'"{citygml_tools_path}" --version')

test_xml_path = "E:\\2_data\\building_3d_opensource\\osaka\\lod2_gml\\51357326_bldg_6697_op.gml"

utils_z.run_cmd(f'"{citygml_tools_path}" to-cityjson {test_xml_path}')

citygml-tools version 2.4.1
(C) 2018-2026 Claus Nagel <claus.nagel@gmail.com>


[09:27:07 INFO] Starting citygml-tools.
[09:27:07 INFO] Executing 'to-cityjson' command.
[09:27:07 INFO] [1|1] Processing file E:\2_data\building_3d_opensource\osaka\lod2_gml\51357326_bldg_6697_op.gml.
[09:27:07 WARN] The input file uses unsupported non-CityGML namespaces: https://www.geospatial.jp/iur/uro/3.2.
[09:27:07 INFO] Non-CityGML content is skipped unless a matching ADE extension has been loaded.
[09:27:07 INFO] Writing output to file E:\2_data\building_3d_opensource\osaka\lod2_gml\51357326_bldg_6697_op.json.
[09:27:07 INFO] Total execution time: 01 s.
[09:27:07 INFO] citygml-tools finished with 1 warning(s).



CompletedProcess(args='"E:\\0_mylib\\citygml-tools-2.4.1\\citygml-tools.bat" to-cityjson E:\\2_data\\building_3d_opensource\\osaka\\lod2_gml\\51357326_bldg_6697_op.gml', returncode=0, stdout="[09:27:07 INFO] Starting citygml-tools.\n[09:27:07 INFO] Executing 'to-cityjson' command.\n[09:27:07 INFO] [1|1] Processing file E:\\2_data\\building_3d_opensource\\osaka\\lod2_gml\\51357326_bldg_6697_op.gml.\n[09:27:07 WARN] The input file uses unsupported non-CityGML namespaces: https://www.geospatial.jp/iur/uro/3.2.\n[09:27:07 INFO] Non-CityGML content is skipped unless a matching ADE extension has been loaded.\n[09:27:07 INFO] Writing output to file E:\\2_data\\building_3d_opensource\\osaka\\lod2_gml\\51357326_bldg_6697_op.json.\n[09:27:07 INFO] Total execution time: 01 s.\n[09:27:07 INFO] citygml-tools finished with 1 warning(s).\n", stderr='')

In [11]:
import json

# test_json_path = test_xml_path.replace('.xml', '.json')
# test_json_path = test_xml_path.replace('.gml', '.json')
test_json_path = "E:\\2_data\\building_3d_opensource\\osaka\\lod2_json\\51357326_bldg_6697_op.json"

with open(test_json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# 查看顶层结构
print(data.keys())

# 查看第一栋建筑的属性
first_building_id = list(data["CityObjects"].keys())[0]
first_building = data["CityObjects"][first_building_id]
print(f"\n建筑ID：{first_building_id}")
print(f"类型：{first_building['type']}")
print(f"属性：{json.dumps(first_building.get('attributes', {}), indent=2, ensure_ascii=False)}")
print(f"几何类型：{[g['type'] for g in first_building.get('geometry', [])]}")

dict_keys(['type', 'version', 'CityObjects', 'transform', 'vertices', 'metadata'])

建筑ID：bldg_cc6496f8-d50c-4797-9b28-901d1b4a07d1
类型：Building
属性：{
  "creationDate": "2023-03-22T00:00:00+08:00",
  "計測周辺長": {
    "value": 26.6842499722,
    "uom": "m"
  },
  "1/2500図郭": 94,
  "区名": "住之江区",
  "町丁目名称": "新北島７丁目",
  "街区番号": 4,
  "measuredHeight": 10.5,
  "class": "3001",
  "usage": "411",
  "storeysAboveGround": 2
}
几何类型：['MultiSurface', 'Solid']


In [6]:
# 查看第一栋建筑的几何信息
geom = first_building["geometry"][0]
print(f"几何type: {geom['type']}")
print(f"LOD: {geom.get('lod')}")
print(f"boundaries层级示例: {str(geom['boundaries'])[:200]}")
print(f"semantics: {geom.get('semantics', {}).get('surfaces', [])[:3]}")
print(f"semantics values: {geom.get('semantics', {}).get('values', [])[:2]}")

几何type: MultiSurface
LOD: 0
boundaries层级示例: [[[0, 0, 0, 0]]]
semantics: []
semantics values: []


In [7]:
attrs = first_building.get("attributes", {})
print("建造年份:", attrs.get("oorspronkelijkbouwjaar"))  # 对应year_built
print("楼层数:", attrs.get("b3_bouwlagen"))               # 对应floor_count
print("屋顶类型:", attrs.get("b3_dak_type"))              # 对应roof_type
print("建筑高度:", attrs.get("b3_h_dak_50p"))             # 对应height，有多个候选

建造年份: None
楼层数: None
屋顶类型: None
建筑高度: None


In [14]:
print("metadata:", data.get("metadata", {}))
print("transform:", data["transform"])
print("第一个顶点原始值:", data["vertices"][0])

metadata: {'fullMetadataUrl': 'https://data.3dbag.nl/metadata/v20250903/metadata.json', 'geographicalExtent': [122084.78125, 486330.21875, -1.9469985961914062, 122609.671875, 486912.84375, 36.496002197265625], 'pointOfContact': {'contactName': '3DBAG Team', 'emailAddress': 'info@3dbag.nl', 'role': 'owner', 'website': 'https://3dbag.nl'}, 'referenceSystem': 'https://www.opengis.net/def/crs/EPSG/0/7415', 'title': '3DBAG', 'version': 'v2025.09.03'}
transform: {'scale': [0.001, 0.001, 0.001], 'translate': [122347.2265625, 486621.53125, 17.275001525878906]}
第一个顶点原始值: [-2248, -63735, -17275]


In [9]:
count = 0
for building_id, building in data['CityObjects'].items():
    if building['type'] == 'BuildingPart':
        for geom in building['geometry']:
            lod = geom['lod']
            print(f"Building {building_id}: LOD {lod}")
        count += 1
        if count >= 10:
            break

Building NL.IMBAG.Pand.0363100012064861-0: LOD 1.2
Building NL.IMBAG.Pand.0363100012064861-0: LOD 1.3
Building NL.IMBAG.Pand.0363100012064861-0: LOD 2.2
Building NL.IMBAG.Pand.0363100012067591-0: LOD 1.2
Building NL.IMBAG.Pand.0363100012067591-0: LOD 1.3
Building NL.IMBAG.Pand.0363100012067591-0: LOD 2.2
Building NL.IMBAG.Pand.0363100012070628-0: LOD 1.2
Building NL.IMBAG.Pand.0363100012070628-0: LOD 1.3
Building NL.IMBAG.Pand.0363100012070628-0: LOD 2.2
Building NL.IMBAG.Pand.0363100012073195-0: LOD 1.2
Building NL.IMBAG.Pand.0363100012073195-0: LOD 1.3
Building NL.IMBAG.Pand.0363100012073195-0: LOD 2.2
Building NL.IMBAG.Pand.0363100012079333-0: LOD 1.2
Building NL.IMBAG.Pand.0363100012079333-0: LOD 1.3
Building NL.IMBAG.Pand.0363100012079333-0: LOD 2.2
Building NL.IMBAG.Pand.0363100012081959-0: LOD 1.2
Building NL.IMBAG.Pand.0363100012081959-0: LOD 1.3
Building NL.IMBAG.Pand.0363100012081959-0: LOD 2.2
Building NL.IMBAG.Pand.0363100012084413-0: LOD 1.2
Building NL.IMBAG.Pand.03631000

In [10]:
for building_id, building in data['CityObjects'].items():
    if building['type'] == 'BuildingPart':
        for geom in building['geometry']:
            if str(geom.get('lod')) == '2.2':
                print(f"ID: {building_id}")
                print(f"boundaries示例: {str(geom['boundaries'])[:150]}")
                print(f"semantics surfaces: {geom.get('semantics', {}).get('surfaces', [])[:3]}")
                print(f"semantics values: {geom.get('semantics', {}).get('values', [])[:2]}")
                print("---")
        break  # 只看第一个BuildingPart

ID: NL.IMBAG.Pand.0363100012064861-0
boundaries示例: [[[[9733, 9745, 9746, 9734, 9735, 9736, 9737, 9747, 9748, 9738]], [[9749, 9738, 9748, 9750]], [[9751, 9735, 9734, 9752]], [[9753, 9737, 9736, 9754]], 
semantics surfaces: [{'type': 'GroundSurface'}, {'on_footprint_edge': True, 'type': 'WallSurface'}, {'on_footprint_edge': False, 'type': 'WallSurface'}]
semantics values: [[0, 1, 1, 1, 1, 1, 2, 1, 1, 1, 2, 1, 2, 2, 2, 2, 2, 2, 1, 3, 4, 5, 6]]
---


In [15]:
# 统计所有对象类型及其数量
object_counts = {}
for obj_id, obj in data['CityObjects'].items():
    obj_type = obj['type']
    object_counts[obj_type] = object_counts.get(obj_type, 0) + 1

print("CityJSON对象类型统计：")
for obj_type, count in sorted(object_counts.items()):
    print(f"  {obj_type}: {count}个")
    
print(f"\n总计: {sum(object_counts.values())}个对象")

CityJSON对象类型统计：
  Building: 318个
  BuildingPart: 320个

总计: 638个对象


In [16]:
lod_sets = {}
for building_id, building in data['CityObjects'].items():
    if building['type'] == 'BuildingPart':
        lods = [str(g['lod']) for g in building.get('geometry', [])]
        key = tuple(sorted(lods))
        lod_sets[key] = lod_sets.get(key, 0) + 1

print("LOD组合分布：")
for k, v in lod_sets.items():
    print(f"  {k}: {v}个BuildingPart")

LOD组合分布：
  ('1.2', '1.3', '2.2'): 320个BuildingPart


In [17]:
surface_types = set()
for building_id, building in data['CityObjects'].items():
    if building['type'] == 'BuildingPart':
        for geom in building['geometry']:
            if str(geom.get('lod')) == '2.2':
                for s in geom.get('semantics', {}).get('surfaces', []):
                    surface_types.add(s.get('type'))

print("所有surface类型：", surface_types)

所有surface类型： {'WallSurface', 'GroundSurface', 'RoofSurface'}


#### 建表lod2.2

In [31]:
# 建表前先创建序列
utils_z.run_sql(f"""
    CREATE SEQUENCE IF NOT EXISTS {lod2_table_name}_seq;
""", conn=conn)

utils_z.run_sql(f"""
    CREATE SEQUENCE IF NOT EXISTS {lod2_surface_table_name}_seq;
""", conn=conn)

# LOD2建筑主表
utils_z.run_sql(f"""
    CREATE TABLE IF NOT EXISTS {lod2_table_name} (
        building_id     VARCHAR PRIMARY KEY 
                        DEFAULT '{city_prefix}_B_' || LPAD(NEXTVAL('{lod2_table_name}_seq')::TEXT, 7, '0'),
        block_id        VARCHAR,
        citygml_id      VARCHAR,
        geom_2d         GEOMETRY(Polygon, {target_srid}),
        height          FLOAT,
        floor_count     INTEGER,
        function        VARCHAR,
        roof_type       VARCHAR,
        year_built      INTEGER
    );
""", conn=conn)

utils_z.run_sql(f"""
    CREATE INDEX IF NOT EXISTS {lod2_table_name}_geom_idx
    ON {lod2_table_name} USING GIST (geom_2d);
""", conn=conn)

# LOD2 surface子表
utils_z.run_sql(f"""
    CREATE TABLE IF NOT EXISTS {lod2_surface_table_name} (
        surface_id      VARCHAR PRIMARY KEY
                        DEFAULT '{city_prefix}_S_' || LPAD(NEXTVAL('{lod2_surface_table_name}_seq')::TEXT, 8, '0'),
        citygml_id      VARCHAR,
        building_id     VARCHAR REFERENCES {lod2_table_name}(building_id),
        surface_type    VARCHAR,
        geom_3d         GEOMETRY(PolygonZ, {target_srid})
    );
""", conn=conn)

utils_z.run_sql(f"""
    CREATE INDEX IF NOT EXISTS {lod2_surface_table_name}_building_idx
    ON {lod2_surface_table_name} (building_id);
""", conn=conn)

print(city_prefix+" LOD2表创建完成")

NL_AM LOD2表创建完成


In [32]:
import traceback

In [33]:
lod2_json_dir = r"E:\2_data\building_3d_opensource\amsterdam\lod2_json"

In [34]:
json_files = [f for f in os.listdir(lod2_json_dir) if f.endswith(".json")]
print(f"共{len(json_files)}个文件待入库")

total = 0
errors = []
building_counter = 1
surface_counter = 1

for i, filename in enumerate(json_files):
    filepath = os.path.join(lod2_json_dir, filename)
    try:
        buildings = cjpar.parse_cityjson_lod2_NL_AM(filepath)
        count, building_counter, surface_counter = cjpar.insert_buildings_lod2(
            buildings, conn,
            lod2_table=lod2_table_name,
            surface_table=lod2_surface_table_name,
            city_prefix=city_prefix,
            target_srid=target_srid,
            source_srid=source_srid,
            building_counter=building_counter,
            surface_counter=surface_counter
        )
        total += count
        if (i + 1) % 50 == 0:
            print(f"入库进度：{i+1}/{len(json_files)}，已入库建筑：{total}，已入库表面：{surface_counter-1}")
    except Exception as e:
        errors.append((filename, str(e)))
        print(f"错误：{filename} -> {e}")
        traceback.print_exc()

print(f"\n完成！共入库建筑：{total}，共入库表面：{surface_counter-1}")

print(f"失败文件数：{len(errors)}")

共146个文件待入库
入库进度：50/146，已入库建筑：34651，已入库表面：1004196
入库进度：100/146，已入库建筑：104709，已入库表面：2113819

完成！共入库建筑：165992，共入库表面：3236008
失败文件数：0


#### 叠合2.2

In [38]:
# 查看入库结果的空间范围，判断srid
for table in ["hamburg_blocks", "vienna_blocks", "amsterdam_blocks"]:
    result = utils_z.run_sql(f"SELECT ST_Extent(geom) FROM {table};", conn=conn, fetch=True)
    print(f"{table}: {result}")

hamburg_blocks: [('BOX(8.487736399999998 53.397543599110314,10.303492599999998 53.92314769917268)',)]
vienna_blocks: [('BOX(16.201112799999997 48.11854529990928,16.5477099 48.308304699863235)',)]
amsterdam_blocks: [('BOX(4.733897599999999 52.28637929909728,5.0360204 52.4283207990906)',)]


In [35]:
# 建立空间索引
utils_z.run_sql(f"""
    CREATE INDEX IF NOT EXISTS buildings_lod2_geom_idx
    ON {lod2_table_name} USING GIST (geom_2d);
""", conn=conn)

print("开始空间叠合...")

开始空间叠合...


In [36]:
utils_z.run_sql(f"""
    UPDATE {lod2_table_name} b
    SET block_id = (
        SELECT bl.block_id
        FROM {block_table_name} bl
        WHERE ST_Within(ST_Centroid(b.geom_2d), bl.geom)
        LIMIT 1
    );
""", conn=conn)
print("空间叠合完成")

空间叠合完成


In [37]:
# 检查匹配情况
result = utils_z.run_sql(f"""
    SELECT
        COUNT(*) AS total,
        COUNT(block_id) AS matched,
        COUNT(*) FILTER (WHERE block_id IS NULL) AS unmatched
    FROM {lod2_table_name};
""", conn=conn, fetch=True)

print(f"总建筑数：{result[0][0]}")
print(f"成功匹配block：{result[0][1]}")
print(f"未匹配block：{result[0][2]}")

总建筑数：165992
成功匹配block：146861
未匹配block：19131


##### debug: 高度等数据呢？

In [27]:
# 检查amsterdam_buildings_lod2表中关键字段的数据完整性
sql_field_check = f"""
    SELECT 
        COUNT(*) as total_rows,
        COUNT(height) as height_count,
        COUNT(floor_count) as floor_count_count,
        COUNT(function) as function_count,
        COUNT(roof_type) as roof_type_count,
        COUNT(year_built) as year_built_count
    FROM {lod2_table_name};
"""
result = utils_z.run_sql(sql_field_check, conn=conn, fetch=True)
row = result[0]

print(f"amsterdam_buildings_lod2 表字段数据统计")
print(f"{'='*50}")
print(f"总行数:        {row[0]}")
print(f"height:        {row[1]:6d} / {row[0]} ({row[1]/row[0]*100:.1f}%)")
print(f"floor_count:   {row[2]:6d} / {row[0]} ({row[2]/row[0]*100:.1f}%)")
print(f"function:      {row[3]:6d} / {row[0]} ({row[3]/row[0]*100:.1f}%)")
print(f"roof_type:     {row[4]:6d} / {row[0]} ({row[4]/row[0]*100:.1f}%)")
print(f"year_built:    {row[5]:6d} / {row[0]} ({row[5]/row[0]*100:.1f}%)")

amsterdam_buildings_lod2 表字段数据统计
总行数:        165992
height:             0 / 165992 (0.0%)
floor_count:        0 / 165992 (0.0%)
function:           0 / 165992 (0.0%)
roof_type:          0 / 165992 (0.0%)
year_built:         0 / 165992 (0.0%)


In [29]:
with open(r"E:\2_data\building_3d_opensource\amsterdam\lod2_json\9-404-704.city.json", "r", encoding="utf-8") as f:
    data = json.load(f)

for obj_id, obj in data["CityObjects"].items():
    if obj["type"] == "Building":
        print("Building属性:", json.dumps(obj.get("attributes", {}), indent=2, ensure_ascii=False)[:500])
        break

for obj_id, obj in data["CityObjects"].items():
    if obj["type"] == "BuildingPart":
        print("BuildingPart属性:", obj.get("attributes", {}))
        break

Building属性: {
  "b3_bag_bag_overlap": 0.0,
  "b3_bouwlagen": 3,
  "b3_dak_type": "slanted",
  "b3_extrusie": "standard",
  "b3_h_dak_50p": 7.802999973297119,
  "b3_h_dak_70p": 8.343999862670898,
  "b3_h_dak_max": 9.442000389099121,
  "b3_h_dak_min": 7.583000183105469,
  "b3_h_maaiveld": -0.5889999866485596,
  "b3_h_nok": 9.56556510925293,
  "b3_is_glas_dak": false,
  "b3_kas_warenhuis": false,
  "b3_kwaliteitsindicator": true,
  "b3_mutatie_ahn3_ahn4": false,
  "b3_mutatie_ahn4_ahn5": false,
  "b3_n_nok": 2
BuildingPart属性: {}


#### 数量检查1.3和2.2

In [25]:
# 检查包含LOD1和LOD2建筑的街区数量
sql_counts = f"""
    SELECT 
        (SELECT COUNT(DISTINCT block_id) FROM {lod2_table_name} WHERE block_id IS NOT NULL) as lod1_blocks,
        (SELECT COUNT(DISTINCT block_id) FROM {lod2_table_name} WHERE block_id IS NOT NULL) as lod2_blocks;
"""
counts = utils_z.run_sql(sql_counts, conn=conn, fetch=True)
print(f"包含LOD1建筑的街区数量: {counts[0][0]}")
print(f"包含LOD2建筑的街区数量: {counts[0][1]}")

包含LOD1建筑的街区数量: 3118
包含LOD2建筑的街区数量: 3118


#### 临时：统一所有数据库的SRID到4326，兼容全球城市
统一存储为EPSG:4326（WGS84经纬度），通用性最强，全球任何城市都适用，可视化也方便。缺点是直接用ST_Area计算面积不准确，需要用ST_Area(ST_Transform(geom, 合适的UTM))或者ST_Area(geom::geography)来保证精度。

* Block转换

In [ ]:
for table, src_srid in [
    ("amsterdam_blocks", 28992),
    
]:
    print(f"开始转换 {table}...")
    
    # 1. 添加新列
    utils_z.run_sql(f"""
        ALTER TABLE {table} ADD COLUMN geom_4326 GEOMETRY(Polygon, 4326);
    """, conn=conn)
    
    # 2. 转换填入
    utils_z.run_sql(f"""
        UPDATE {table} SET geom_4326 = ST_Transform(geom, 4326);
    """, conn=conn)
    
    # 3. 删除旧列，重命名新列
    utils_z.run_sql(f"""
        ALTER TABLE {table} DROP COLUMN geom;
    """, conn=conn)
    utils_z.run_sql(f"""
        ALTER TABLE {table} RENAME COLUMN geom_4326 TO geom;
    """, conn=conn)
    
    # 4. 重建空间索引
    utils_z.run_sql(f"""
        DROP INDEX IF EXISTS {table}_geom_idx;
    """, conn=conn)
    utils_z.run_sql(f"""
        CREATE INDEX {table}_geom_idx ON {table} USING GIST (geom);
    """, conn=conn)
    
    print(f"{table} 转换完成")

开始转换 amsterdam_blocks...


In [6]:
# 验证
for table in ["hamburg_blocks", "vienna_blocks", "amsterdam_blocks"]:
    result = utils_z.run_sql(f"SELECT ST_Extent(geom) FROM {table};", conn=conn, fetch=True)
    print(f"{table}: {result}")

hamburg_blocks: [('BOX(8.487736399999998 53.397543599110314,10.303492599999998 53.92314769917268)',)]
vienna_blocks: [('BOX(16.201112799999997 48.11854529990928,16.5477099 48.308304699863235)',)]
amsterdam_blocks: [('BOX(4.733897599999999 52.28637929909728,5.0360204 52.4283207990906)',)]


In [7]:
for table, col in [
    ("hamburg_buildings_lod2", "geom_2d"),
    ("vienna_buildings_lod2",  "geom_2d"),
]:
    result = utils_z.run_sql(f"""
        SELECT ST_Extent({col}) FROM {table};
    """, conn=conn, fetch=True)
    print(f"{table}: {result}")

hamburg_buildings_lod2: [('BOX(466355.917 5917227.923,587611.77 5975921.492)',)]
vienna_buildings_lod2: [('BOX(588193.6892276492 5330586.776875203,617274.0278011339 5352664.984628345)',)]
